In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#Load the ratings dataframe from the parquet tables we created
artist_tags = pd.read_parquet('./data/mb_artist_tag.parquet')
album_tags = pd.read_parquet('./data/mb_album_tag.parquet')
album_label = pd.read_parquet('./data/mb_album_label.parquet')

# Load the complete album and artist universes up-front — these define the master
# row indices so that albums/artists with no tags are preserved as zero-rows in
# every sparse matrix rather than being silently dropped.
unique_album_ids = pd.Index(pd.read_parquet('./data/mb_album.parquet', columns=['id'])['id'].sort_values())
unique_artist_ids = pd.Index(pd.read_parquet('./data/mb_artist.parquet', columns=['id'])['id'].sort_values())

# Verify the loads
dataframes = {
    "Artist Tags": artist_tags,
    "Album Tags": album_tags,
    "Album Label": album_label
}

for name, df in dataframes.items():
    print(f"✅ {name}: {df.shape[0]:,} rows loaded.")

print(f"✅ Full album universe: {len(unique_album_ids):,} albums")
print(f"✅ Full artist universe: {len(unique_artist_ids):,} artists")

In [ ]:
print("1. Filtering out rare artist tags...")
# Group by tag to count occurrences and filter out tags appearing < 10 times
artist_tag_counts = artist_tags.groupby('tag_id').size()
popular_artist_tags = artist_tag_counts[artist_tag_counts >= 10].index
artist_tags_filtered = artist_tags[artist_tags['tag_id'].isin(popular_artist_tags)].copy()

print("2. Normalizing artist tag weights...")
# Calculate relative weights so an artist's profile bounds between 0.0 and 1.0
artist_totals = artist_tags_filtered.groupby('artist_id')['tag_count'].transform('sum')
artist_tags_filtered['tag_weight'] = (artist_tags_filtered['tag_count'] / artist_totals).astype('float32')

print("3. Generating category codes...")
# Tags get dense sequential codes within the filtered set
artist_tags_filtered['tag_code'] = artist_tags_filtered['tag_id'].astype('category').cat.codes
unique_artist_tag_ids = artist_tags_filtered['tag_id'].astype('category').cat.categories

# Artist rows are mapped against the FULL artist universe (unique_artist_ids, loaded above)
# so artists without popular tags still occupy a row as zeros — no artists dropped
artist_tags_filtered['artist_code'] = unique_artist_ids.get_indexer(artist_tags_filtered['artist_id'])

print("4. Building Artist Sparse Matrix via COOrdinate mapping...")
# Extract coordinate arrays
artist_row_indices = artist_tags_filtered['artist_code'].values
artist_col_indices = artist_tags_filtered['tag_code'].values
artist_weights = artist_tags_filtered['tag_weight'].values

# Construct the sparse matrix directly
X_artist_tags_sparse = csr_matrix(
    (artist_weights, (artist_row_indices, artist_col_indices)), 
    shape=(len(unique_artist_ids), len(unique_artist_tag_ids))
)

print(f"🚀 Done! Artist Matrix Shape: {X_artist_tags_sparse.shape}")
print(f"Non-zero elements tracked: {X_artist_tags_sparse.nnz}")

In [ ]:
print("1. Filtering out rare tags...")
# Group by tag to count occurrences and filter rare ones (>= 10)
tag_counts = album_tags.groupby('tag_id').size()
popular_tags = tag_counts[tag_counts >= 10].index
album_tags_filtered = album_tags[album_tags['tag_id'].isin(popular_tags)].copy()

print("2. Normalizing tag weights...")
# Calculate relative weights using transform('sum') on the filtered set
album_totals = album_tags_filtered.groupby('album_id')['tag_count'].transform('sum')
album_tags_filtered['tag_weight'] = (album_tags_filtered['tag_count'] / album_totals).astype('float32')

print("3. Generating category codes...")
# Tags get dense sequential codes within the filtered set
album_tags_filtered['tag_code'] = album_tags_filtered['tag_id'].astype('category').cat.codes
unique_tag_ids = album_tags_filtered['tag_id'].astype('category').cat.categories

# Album rows are mapped against the FULL album universe (unique_album_ids, loaded above)
# so albums without popular tags still occupy a row as zeros — no albums dropped
album_tags_filtered['album_code'] = unique_album_ids.get_indexer(album_tags_filtered['album_id'])

print("4. Building Sparse Matrix instantly via COOrdinate mapping...")
# Extract our rows, columns, and data values as raw numpy arrays
row_indices = album_tags_filtered['album_code'].values
col_indices = album_tags_filtered['tag_code'].values
weights = album_tags_filtered['tag_weight'].values

# Create the CSR sparse matrix directly from the coordinates
# Dimensions: Number of unique albums x Number of unique popular tags
X_album_tags_sparse = csr_matrix(
    (weights, (row_indices, col_indices)), 
    shape=(len(unique_album_ids), len(unique_tag_ids))
)

print(f"🚀 Done! Matrix Shape: {X_album_tags_sparse.shape}")
print(f"Non-zero data elements tracked: {X_album_tags_sparse.nnz}")

In [ ]:
print("1. Filtering out rare labels...")
album_label['label_type'] = album_label['label_type'].fillna(0.0).astype('int32')
label_counts = album_label.groupby('label_id').size()
popular_labels = label_counts[label_counts >= 10].index
label_filtered = album_label[album_label['label_id'].isin(popular_labels)].copy()

print("2. Normalizing label weights...")
label_totals = label_filtered.groupby('album_id')['tag_count'].transform('sum')
label_filtered['label_weight'] = (label_filtered['tag_count'] / label_totals).astype('float32')

print("3. Generating category codes...")
# CRUCIAL: We map album_id based on the exact same universe of unique_album_ids from your tags step
# This guarantees that Row 5 in this matrix is the exact same album as Row 5 in your tag matrix!
label_filtered['album_id'] = pd.Categorical(label_filtered['album_id'], categories=unique_album_ids)
label_filtered = label_filtered.dropna(subset=['album_id']).copy() # Drop any albums that didn't have tags

label_filtered['album_code'] = label_filtered['album_id'].cat.codes
label_filtered['label_code'] = label_filtered['label_id'].astype('category').cat.codes
label_filtered['type_code'] = label_filtered['label_type'].astype('category').cat.codes

unique_label_ids = label_filtered['label_id'].astype('category').cat.categories
unique_type_ids = label_filtered['label_type'].astype('category').cat.categories

print("4. Building Sparse Matrices...")
# Label Matrix
X_album_labels_sparse = csr_matrix(
    (label_filtered['label_weight'].values, (label_filtered['album_code'].values, label_filtered['label_code'].values)),
    shape=(len(unique_album_ids), len(unique_label_ids))
)

# Label Type Matrix (Binary indicator)
# We give it a uniform weight of 1.0, then normalize the rows so they scale between 0 and 1
ones = np.ones(len(label_filtered), dtype='float32')
X_album_types_sparse = csr_matrix(
    (ones, (label_filtered['album_code'].values, label_filtered['type_code'].values)),
    shape=(len(unique_album_ids), len(unique_type_ids))
)
# Normalize row-wise so multiple labels per album don't push values past 1.0
from sklearn.preprocessing import normalize
X_album_types_sparse = normalize(X_album_types_sparse, norm='l1', axis=1)

print(f"🚀 Labels Shape: {X_album_labels_sparse.shape} | Types Shape: {X_album_types_sparse.shape}")

In [ ]:
# Set up a clean, professional aesthetic for the report
sns.set_theme(style="whitegrid")

# Create a spacious 3x2 grid of subplots
fig, axes = plt.subplots(3, 2, figsize=(16, 18))

# ----------------------------------------------------------------------
# ROW 1: ALBUM TAGS MATRIX VISUALIZATION
# ----------------------------------------------------------------------
# Left: Profile Complexity (Non-zero tags per album)
album_tags_per_row = X_album_tags_sparse.getnnz(axis=1)
sns.histplot(album_tags_per_row, bins=range(0, 35), ax=axes[0, 0], color='#4A90E2', kde=True)
axes[0, 0].set_title('Album Tags: Profile Complexity (Tags per Album)', fontsize=12, weight='bold')
axes[0, 0].set_xlabel('Number of Unique Tags on a Single Album')
axes[0, 0].set_ylabel('Count of Albums')
axes[0, 0].set_xlim(0, 30)

# Right: Long-Tail Distribution (Album Tag Popularity)
album_tag_popularity = np.sort(X_album_tags_sparse.getnnz(axis=0))[::-1]
axes[0, 1].plot(album_tag_popularity, color='#4A90E2', linewidth=2.5)
axes[0, 1].fill_between(range(len(album_tag_popularity)), album_tag_popularity, color='#4A90E2', alpha=0.25)
axes[0, 1].set_yscale('log')
axes[0, 1].set_title('Album Tags: Long-Tail Feature Popularity', fontsize=12, weight='bold')
axes[0, 1].set_xlabel('Tag Index (Sorted by Global Popularity)')
axes[0, 1].set_ylabel('Number of Albums Sharing Tag (Log Scale)')


# ----------------------------------------------------------------------
# ROW 2: ALBUM LABELS MATRIX VISUALIZATION
# ----------------------------------------------------------------------
# Left: Profile Complexity (Non-zero labels per album)
album_labels_per_row = X_album_labels_sparse.getnnz(axis=1)
sns.histplot(album_labels_per_row, bins=range(0, 10), ax=axes[1, 0], color='#E056FD', kde=False)
axes[1, 0].set_title('Album Labels: Profile Complexity (Labels per Album)', fontsize=12, weight='bold')
axes[1, 0].set_xlabel('Number of Unique Record Labels on a Single Album')
axes[1, 0].set_ylabel('Count of Albums')
axes[1, 0].set_xlim(0, 6)

# Right: Long-Tail Distribution (Album Label Popularity)
album_label_popularity = np.sort(X_album_labels_sparse.getnnz(axis=0))[::-1]
axes[1, 1].plot(album_label_popularity, color='#E056FD', linewidth=2.5)
axes[1, 1].fill_between(range(len(album_label_popularity)), album_label_popularity, color='#E056FD', alpha=0.25)
axes[1, 1].set_yscale('log')
axes[1, 1].set_title('Album Labels: Long-Tail Feature Popularity', fontsize=12, weight='bold')
axes[1, 1].set_xlabel('Label Index (Sorted by Global Popularity)')
axes[1, 1].set_ylabel('Number of Albums Sharing Label (Log Scale)')


# ----------------------------------------------------------------------
# ROW 3: ARTIST TAGS MATRIX VISUALIZATION
# ----------------------------------------------------------------------
# Left: Profile Complexity (Non-zero tags per artist)
artist_tags_per_row = X_artist_tags_sparse.getnnz(axis=1)
sns.histplot(artist_tags_per_row, bins=range(0, 45), ax=axes[2, 0], color='#10AC84', kde=True)
axes[2, 0].set_title('Artist Tags: Profile Complexity (Tags per Artist)', fontsize=12, weight='bold')
axes[2, 0].set_xlabel('Number of Unique Tags on a Single Artist')
axes[2, 0].set_ylabel('Count of Artists')
axes[2, 0].set_xlim(0, 40)

# Right: Long-Tail Distribution (Artist Tag Popularity)
artist_tag_popularity = np.sort(X_artist_tags_sparse.getnnz(axis=0))[::-1]
axes[2, 1].plot(artist_tag_popularity, color='#10AC84', linewidth=2.5)
axes[2, 1].fill_between(range(len(artist_tag_popularity)), artist_tag_popularity, color='#10AC84', alpha=0.25)
axes[2, 1].set_yscale('log')
axes[2, 1].set_title('Artist Tags: Long-Tail Feature Popularity', fontsize=12, weight='bold')
axes[2, 1].set_xlabel('Tag Index (Sorted by Global Popularity)')
axes[2, 1].set_ylabel('Number of Artists Sharing Tag (Log Scale)')


# Adjust subplots to ensure labels and titles are clearly readable and un-truncated
plt.tight_layout()

# Save the full visual matrix directly to your repository folder
plt.savefig('sparse_features_structural_analysis.png', dpi=300)

In [ ]:
import os
import pickle

os.makedirs('data/features', exist_ok=True)

# Save your ID mappings so other notebooks can read the row alignments
with open('data/features/artist_ids.pkl', 'wb') as f:
    pickle.dump(unique_artist_ids.tolist(), f)

with open('data/features/album_ids.pkl', 'wb') as f:
    pickle.dump(unique_album_ids.tolist(), f)

In [ ]:
import os
import pickle
from scipy.sparse import save_npz

# Ensure the target directory exists
os.makedirs('data/features', exist_ok=True)

print("Saving Sparse Matrices to data/features/...")
save_npz('data/features/artist_tags_matrix.npz', X_artist_tags_sparse)
save_npz('data/features/album_tags_matrix.npz', X_album_tags_sparse)
save_npz('data/features/album_labels_matrix.npz', X_album_labels_sparse)
save_npz('data/features/album_types_matrix.npz', X_album_types_sparse)

print("Saving Index IDs to data/features/...")
with open('data/features/artist_ids.pkl', 'wb') as f:
    pickle.dump(unique_artist_ids.tolist(), f)

with open('data/features/album_ids.pkl', 'wb') as f:
    pickle.dump(unique_album_ids.tolist(), f)

print("🎉 Metadata successfully exported to data/features/")